# 52 - Random Forest (Sale y Rent)

Entrenamiento de `RandomForestRegressor` para predecir `log_precio` en `final_sale.csv` y `final_rent.csv`, con validacion cruzada K-Fold=5 sobre train, metricas completas y bloque SHAP.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
TARGET_COL = "log_precio"

RF_PARAMS = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

print(f"SHAP disponible: {SHAP_AVAILABLE}")

In [ ]:
def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "gold").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/gold")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
OUTPUT_DIR = PROJECT_ROOT / "data" / "ML" / "random_forest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATHS = {
    "sale": GOLD_DIR / "final_sale.csv",
    "rent": GOLD_DIR / "final_rent.csv",
}

for name, path in DATASET_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"No se encontro {name}: {path}")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
EXCLUDE_COLS = {
    "codigo_inmueble",
    "precio",
    "precio_m2",
    "precio_m2_municipio_media",
    "planta",
    "inv_distancia_min_playa_km",
    "inv_distancia_min_supermercado_km",
    "inv_distancia_min_colegio_km",
    "superficie_construida_m2_2",
    "numero_banos_2",
    "numero_dormitorios_2",
    "habitaciones_2",
}

BASE_FEATURES = [
    "superficie_construida_m2",
    "numero_dormitorios",
    "numero_banos",
    "tipologia",
    "latitud",
    "longitud",
    "es_exterior",
    "tiene_ascensor",
    "tiene_garaje",
    "obra_nueva",
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
    "log_superficie_construida_m2",
    "ratio_dormitorios_superficie",
    "ratio_banos_superficie",
    "habitaciones",
    "ratio_habitaciones_superficie",
    "interaccion_superficie_banos",
    "interaccion_superficie_habitaciones",
    "planta_num",
    "interaccion_planta_sin_ascensor_piso",
    "latitud_2",
    "longitud_2",
    "interaccion_latitud_longitud",
    "distancia_centro_municipio_km",
    "score_cercania_servicios",
    "tipologia_unificada_piso",
    "tipologia_unificada_unifamiliar",
    "subtipologia_casaDePueblo",
    "subtipologia_countryHouse",
    "subtipologia_desconocido",
    "subtipologia_duplex",
    "subtipologia_independantHouse",
    "subtipologia_penthouse",
    "subtipologia_semidetachedHouse",
    "subtipologia_studio",
    "subtipologia_terracedHouse",
    "provincia_Cantabria",
    "provincia_Vizcaya",
]

def build_feature_list(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    prefix_cols = [c for c in df.columns if c.startswith("municipio_") or c.startswith("distrito_")]

    requested = BASE_FEATURES + prefix_cols
    requested = list(dict.fromkeys(requested))

    allowed = [c for c in requested if c in df.columns]
    allowed = [c for c in allowed if c not in EXCLUDE_COLS and c != TARGET_COL]

    missing = [c for c in BASE_FEATURES if c not in df.columns]
    return allowed, missing

def adjusted_r2(r2: float, n: int, p: int) -> float:
    if n <= p + 1:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def compute_metrics(y_true: pd.Series, y_pred: np.ndarray, p: int) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MSE": float(mse),
        "RMSE": rmse,
        "MAE": float(mae),
        "MAPE": float(mape),
        "R2": float(r2),
        "R2_ajustado": float(adjusted_r2(r2, len(y_true), p)),
    }

def preprocess_train_test(X_train: pd.DataFrame, X_test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    X_train = X_train.copy()
    X_test = X_test.copy()

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]

    if num_cols:
        medians = X_train[num_cols].median()
        X_train[num_cols] = X_train[num_cols].fillna(medians)
        X_test[num_cols] = X_test[num_cols].fillna(medians)

    if cat_cols:
        X_train[cat_cols] = X_train[cat_cols].fillna("desconocido").astype(str)
        X_test[cat_cols] = X_test[cat_cols].fillna("desconocido").astype(str)

        X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False, dtype=int)
        X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False, dtype=int)

    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
    return X_train, X_test

In [ ]:
all_results = []

for dataset_name, dataset_path in DATASET_PATHS.items():
    print("\n" + "#" * 120)
    print(f"Dataset: {dataset_name} | Archivo: {dataset_path.name}")

    df = pd.read_csv(dataset_path)

    if TARGET_COL not in df.columns:
        raise ValueError(f"No existe target {TARGET_COL} en {dataset_name}")

    features, missing_base = build_feature_list(df)
    print(f"Filas/columnas originales: {df.shape}")
    print(f"Features seleccionadas: {len(features)}")
    print(f"Base features ausentes: {missing_base}")

    # 1) Eliminar filas con target nulo
    df = df[df[TARGET_COL].notna()].copy()

    # 2) Separar X e y
    X = df[features].copy()
    y = df[TARGET_COL].copy()

    # 3) Split 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # 4) Missing values + encoding (sin scaler)
    X_train_proc, X_test_proc = preprocess_train_test(X_train, X_test)

    p_features = X_train_proc.shape[1]
    print(f"Features realmente usadas por el modelo: {p_features}")

    # 5) Modelo
    rf = RandomForestRegressor(**RF_PARAMS)

    # 6) CV KFold=5 SOLO sobre train
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "mape": "neg_mean_absolute_percentage_error",
        "r2": "r2",
    }

    cv_out = cross_validate(
        rf,
        X_train_proc,
        y_train,
        cv=kf,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    cv_metrics = {
        "CV_MSE": float((-cv_out["test_mse"]).mean()),
        "CV_RMSE": float(np.sqrt((-cv_out["test_mse"]).mean())),
        "CV_MAE": float((-cv_out["test_mae"]).mean()),
        "CV_MAPE": float((-cv_out["test_mape"]).mean()),
        "CV_R2": float((cv_out["test_r2"]).mean()),
    }

    # 7) Fit final en train completo
    rf.fit(X_train_proc, y_train)

    # 8) Predicciones
    pred_train = rf.predict(X_train_proc)
    pred_test = rf.predict(X_test_proc)

    # 9) Metricas train y test
    metrics_train = compute_metrics(y_train, pred_train, p_features)
    metrics_test = compute_metrics(y_test, pred_test, p_features)

    metrics_table = pd.DataFrame([
        {"split": "train", **metrics_train},
        {"split": "test", **metrics_test},
    ])

    print("\nMetricas Train/Test:")
    print(metrics_table.to_string(index=False))

    cv_table = pd.DataFrame([cv_metrics])
    print("\nMetricas CV (KFold=5 sobre train):")
    print(cv_table.to_string(index=False))

    # Bloque de sobreajuste
    overfit = {
        "dataset": dataset_name,
        "delta_RMSE_test_minus_train": metrics_test["RMSE"] - metrics_train["RMSE"],
        "ratio_RMSE_test_train": metrics_test["RMSE"] / max(metrics_train["RMSE"], 1e-9),
        "delta_R2_train_minus_test": metrics_train["R2"] - metrics_test["R2"],
        "delta_MAPE_test_minus_train": metrics_test["MAPE"] - metrics_train["MAPE"],
    }

    print("\nChequeo rapido de sobreajuste:")
    print(pd.DataFrame([overfit]).to_string(index=False))

    # Feature importances
    importances = (
        pd.DataFrame({"feature": X_train_proc.columns, "importance": rf.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    print("\nTop 20 feature importances:")
    print(importances.head(20).to_string(index=False))

    # Guardar resultados dataset
    ds_out = OUTPUT_DIR / dataset_name
    ds_out.mkdir(parents=True, exist_ok=True)

    metrics_table.to_csv(ds_out / "metrics_train_test.csv", index=False)
    cv_table.to_csv(ds_out / "metrics_cv.csv", index=False)
    pd.DataFrame([overfit]).to_csv(ds_out / "overfit_check.csv", index=False)
    importances.to_csv(ds_out / "feature_importances.csv", index=False)

    # 10) SHAP
    if SHAP_AVAILABLE:
        # Muestra razonable para no hacer SHAP innecesariamente pesado
        shap_n = min(300, len(X_test_proc))
        X_shap = X_test_proc.sample(shap_n, random_state=RANDOM_STATE)

        explainer = shap.TreeExplainer(rf)
        shap_values = explainer.shap_values(X_shap)

        # Summary plot
        plt.figure(figsize=(11, 7))
        shap.summary_plot(shap_values, X_shap, show=False)
        plt.tight_layout()
        plt.savefig(ds_out / "shap_summary.png", dpi=140)
        plt.show()
        plt.close()

        # Bar plot global
        plt.figure(figsize=(11, 7))
        shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
        plt.tight_layout()
        plt.savefig(ds_out / "shap_bar.png", dpi=140)
        plt.show()
        plt.close()

        print(f"SHAP guardado en: {ds_out}")
    else:
        print("SHAP no disponible en este entorno. Para activarlo, instala el paquete 'shap' y vuelve a ejecutar este bloque.")

    all_results.append({
        "dataset": dataset_name,
        "n_rows": len(df),
        "n_features_model": p_features,
        "train_MSE": metrics_train["MSE"],
        "train_RMSE": metrics_train["RMSE"],
        "train_MAE": metrics_train["MAE"],
        "train_MAPE": metrics_train["MAPE"],
        "train_R2": metrics_train["R2"],
        "train_R2_ajustado": metrics_train["R2_ajustado"],
        "test_MSE": metrics_test["MSE"],
        "test_RMSE": metrics_test["RMSE"],
        "test_MAE": metrics_test["MAE"],
        "test_MAPE": metrics_test["MAPE"],
        "test_R2": metrics_test["R2"],
        "test_R2_ajustado": metrics_test["R2_ajustado"],
        "CV_MSE": cv_metrics["CV_MSE"],
        "CV_RMSE": cv_metrics["CV_RMSE"],
        "CV_MAE": cv_metrics["CV_MAE"],
        "CV_MAPE": cv_metrics["CV_MAPE"],
        "CV_R2": cv_metrics["CV_R2"],
        "delta_RMSE_test_minus_train": overfit["delta_RMSE_test_minus_train"],
        "ratio_RMSE_test_train": overfit["ratio_RMSE_test_train"],
        "delta_R2_train_minus_test": overfit["delta_R2_train_minus_test"],
    })

summary = pd.DataFrame(all_results).sort_values("dataset").reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / "summary_all_datasets.csv", index=False)

print("\n" + "#" * 120)
print("Resumen final (sale + rent):")
print(summary.to_string(index=False))
print(f"\nResumen guardado en: {OUTPUT_DIR / 'summary_all_datasets.csv'}")